# Lesson 2 — Valid prices come from the market's own bands

The scalar tick size is gone. Each market publishes price_ranges — start, end and step — and the step depends on where in the range the price sits, with finer ticks at the edges than in the centre.

**The rule.** `valid(p) means p = start + k steps within the band containing p`

**When it holds.** For every order price. Snapping is directional: a buy rounds up and a sell rounds down, never toward the price that flatters the trade.

**When it fails.** Reading the structure NAME instead of the bands. A client that switches on 'linear_cent' prices every market on the next structure wrong, and the exchange rejects the order rather than correcting it.

| | |
|---|---|
| Lesson id | `grid` |
| Pane it appears on | `books` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/grid.py` |
| Tests that go red if it stops being true | `tests/test_coherence_grid.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The bands a recorded market actually published

In [ ]:
from modules.coherence.kernel.grid import GridError, parse_price_ranges

row = fixture("markets_ladder")["body"]["markets"][0]
recorded_ticker = row["ticker"]
grid = parse_price_ranges(row["price_ranges"], row["price_level_structure"])

print(f"  {recorded_ticker} publishes structure {grid.structure!r}")
for band in grid.bands:
    print(f"    [{band.start}, {band.end}] step {band.step}")
print(f"  finest step {grid.finest_step}")
print()
print("  Every market in the captured fixtures sits on one flat cent band. That is a")
print("  fact about what was recorded, not a licence to assume it.")

## 2. A banded grid: the step depends on where the price is

In [ ]:
# A banded payload of the shape the venue documents: finer ticks at the edges,
# where a cent is a large fraction of the contract. Written out here rather than
# recorded, because none of the captured markets carries more than one band.
BANDED = [
    {"start": "0.0000", "end": "0.1000", "step": "0.0010"},
    {"start": "0.1000", "end": "0.9000", "step": "0.0100"},
    {"start": "0.9000", "end": "1.0000", "step": "0.0010"},
]
banded = parse_price_ranges(BANDED, "edges_finer_than_the_centre")

print("  price     band step   valid   buy snaps to   sell snaps to")
for probe in ("0.0500", "0.0505", "0.4250", "0.9505"):
    price = Decimal(probe)
    step = banded.band_for(price).step
    print(
        f"  {price}    {step}      {str(banded.is_valid(price)):<6}  "
        f"{banded.snap(price, 'buy')}         {banded.snap(price, 'sell')}"
    )
print()
print("  The step depends on WHERE the price is, so snapping is a lookup, not a division.")

## 3. Snapping is directional, and never optimistic

In [ ]:
price = Decimal("0.4250")
print(f"  {price} is off the grid in the centre band.")
print(f"    a buy  snaps UP   to {banded.snap(price, 'buy')} — you must be willing to pay the next valid price")
print(f"    a sell snaps DOWN to {banded.snap(price, 'sell')} — you must be willing to accept the next one")
print()
print("  Neither ever moves toward the price that would flatter the trade. A leg snapped")
print("  the wrong way turns a positive edge negative while still looking executable.")

## 4. The failure: reading the structure name instead of the bands

In [ ]:
cent = parse_price_ranges([{"start": "0.0000", "end": "1.0000", "step": "0.0100"}], "linear_cent")
edge_price = Decimal("0.0505")
by_bands = banded.snap(edge_price, "buy")
by_name = cent.snap(edge_price, "buy")

print(f"  reading the market's own bands prices {edge_price} at {by_bands}")
print(f"  switching on the STRUCTURE NAME instead sends {by_name}")
print(f"  that is {by_name - by_bands} per contract given away on a contract worth {edge_price}")
print()
print("  There are a dozen structure names and new ones arrive by changelog. A client")
print("  that reads the bands is correct for structures that do not exist yet.")
print()
try:
    parse_price_ranges(None, "linear_cent")
except GridError as exc:
    print(f"  With no bands at all the engine refuses rather than defaulting: {exc}")